In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

df = yf.download("SPY", period="2y", interval="4h", multi_level_index=False)

class ATR:
    def __init__(self, df, period=14):
        self.df = df
        self.period = period

    def calculate(self):
        tr = np.maximum(self.df['High'] - self.df['Low'],
                        np.maximum(abs(self.df['High'] - self.df['Close'].shift(1)),
                                   abs(self.df['Low'] - self.df['Close'].shift(1))))
        return tr.ewm(com=self.period - 1, adjust=False).mean()

[*********************100%***********************]  1 of 1 completed


In [ ]:
class SuperTrend:
    def __init__(self, df, period=10, multiplier=3):
        self.df = df
        self.period = period
        self.multiplier = multiplier

    def calculate(self):
        self.df['ATR'] = ATR(self.df).calculate()
        SuperTrend = (self.df["High"] + self.df["Low"]) / 2 - self.df['ATR'] * self.multiplier
        self.df['SuperTrend'] = SuperTrend.cummax()
        return self.df['SuperTrend']


2024-06-10 13:30:00-04:00    529.829956
2024-06-11 09:30:00-04:00    529.829956
2024-06-11 13:30:00-04:00    529.829956
2024-06-12 09:30:00-04:00    529.829956
2024-06-12 13:30:00-04:00    530.159831
                                ...    
2026-06-08 09:30:00-04:00    755.459193
2026-06-08 13:30:00-04:00    755.459193
2026-06-09 09:30:00-04:00    755.459193
2026-06-09 13:30:00-04:00    755.459193
2026-06-10 09:30:00-04:00    755.459193
Name: SuperTrend, Length: 996, dtype: float64
                                Close        High         Low        Open  \
2024-06-10 09:30:00-04:00  534.309998  535.140015  534.260010  534.390015   
2024-06-10 13:30:00-04:00  535.659973  535.989990  534.229980  534.318787   
2024-06-11 09:30:00-04:00  534.250000  534.809998  532.049988  533.719971   
2024-06-11 13:30:00-04:00  536.909973  537.010010  534.195007  534.243530   
2024-06-12 09:30:00-04:00  542.919983  543.960022  541.244995  541.359985   
...                               ...         ...   

In [ ]:
class ADX:
    def __init__(self, df, period=14):
        self.df = df
        self.period = period
    
    def calculate(self):
        self.df["+DM"] = (self.df["High"] - self.df["High"].shift(1)).clip(0).rolling(self.period).mean()
        self.df["-DM"] = (self.df["Low"] - self.df["Low"].shift(1)).clip(0).rolling(self.period).mean()
        self.df["+DI"] = 100 * self.df["+DM"] / ATR(self.df).calculate()
        self.df["-DI"] = 100 * self.df["-DM"] / ATR(self.df).calculate()
        self.df["DX"] = 100 * abs(self.df["+DI"] - self.df["-DI"]) / (self.df["+DI"] + self.df["-DI"])
        self.df["ADX"] = self.df["DX"].ewm(com=self.period-1, adjust=False).mean()
        return self.df.iloc[0:self.period-1, self.df.columns.get_loc("ADX")].dropna()


,Close,High,Low,Open,Volume,ATR,SuperTrend,+DM,-DM,+DI,-DI,DX,ADX
2024-06-10 09:30:00-04:00,534.309998,535.140015,534.260010,534.390015,2113542,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-06-10 13:30:00-04:00,535.659973,535.989990,534.229980,534.318787,17048087,1.760010,529.829956,NaN,NaN,NaN,NaN,NaN,NaN
2024-06-11 09:30:00-04:00,534.250000,534.809998,532.049988,533.719971,15051069,3.486654,529.829956,NaN,NaN,NaN,NaN,NaN,NaN
2024-06-11 13:30:00-04:00,536.909973,537.010010,534.195007,534.243530,15246878,2.859779,529.829956,NaN,NaN,NaN,NaN,NaN,NaN
2024-06-12 09:30:00-04:00,542.919983,543.960022,541.244995,541.359985,22696815,6.770698,529.829956,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-06-08 09:30:00-04:00,742.030029,745.340027,740.080017,743.359985,26089816,8.079625,755.459193,0.591793,1.502145,7.324505,18.591766,43.475627,12.155315
2026-06-08 13:30:00-04:00,739.309998,742.450012,738.190002,742.010010,15726460,4.514651,755.459193,0.571790,1.154288,12.665219,25.567607,33.746884,13.697570
2026-06-09 09:30:00-04:00,730.760010,746.900024,722.590027,743.630005,48075104,22.990308,755.459193,0.680363,1.112143,2.959348,4.837445,24.088063,14.439748
2026-06-09 13:30:00-04:00,737.059998,737.690002,730.210022,730.780029,26516509,8.514002,755.459193,0.680363,1.625000,7.991110,19.086206,40.975614,16.335167


In [12]:
class RSI:
    def __init__(self, df, period=14):
        self.df = df
        self.period = period
    
    def calculate(self):
        self.df["Delta"] = self.df['Close'].diff()
        self.df["Gain"] = self.df["Delta"].clip(lower=0)
        self.df["Loss"] = -self.df["Delta"].clip(upper=0)
        self.df["Avg_Gain"] = self.df["Gain"].ewm(com=self.period-1, adjust=False).mean()
        self.df["Avg_Loss"] = self.df["Loss"].ewm(com=self.period-1, adjust=False).mean()
        self.df["RS"] = self.df["Avg_Gain"] / self.df["Avg_Loss"]
        self.df["RSI"] = 100 - (100 / (1 + self.df["RS"]))
        return self.df.iloc[0:self.period-1, self.df.columns.get_loc("RSI")].dropna()
    
    print(RSI(df).calculate())

2024-06-10 13:30:00-04:00    100.000000
2024-06-11 09:30:00-04:00     92.563298
2024-06-11 13:30:00-04:00     93.539418
2024-06-12 09:30:00-04:00     95.103313
2024-06-12 13:30:00-04:00     88.985669
2024-06-13 09:30:00-04:00     89.167977
2024-06-13 13:30:00-04:00     89.503191
2024-06-14 09:30:00-04:00     88.182612
2024-06-14 13:30:00-04:00     88.557969
2024-06-17 09:30:00-04:00     90.408551
2024-06-17 13:30:00-04:00     90.717578
2024-06-18 09:30:00-04:00     91.174693
Name: RSI, dtype: float64


In [11]:
print(SuperTrend(df).calculate().dropna())
print(ADX(df).calculate())
print(RSI(df).calculate())

2024-06-10 13:30:00-04:00    529.829956
2024-06-11 09:30:00-04:00    529.829956
2024-06-11 13:30:00-04:00    529.829956
2024-06-12 09:30:00-04:00    529.829956
2024-06-12 13:30:00-04:00    530.159831
                                ...    
2026-06-08 09:30:00-04:00    755.459193
2026-06-08 13:30:00-04:00    755.459193
2026-06-09 09:30:00-04:00    755.459193
2026-06-09 13:30:00-04:00    755.459193
2026-06-10 09:30:00-04:00    755.459193
Name: SuperTrend, Length: 996, dtype: float64
Series([], Name: ADX, dtype: float64)
2024-06-10 13:30:00-04:00    100.000000
2024-06-11 09:30:00-04:00     92.563298
2024-06-11 13:30:00-04:00     93.539418
2024-06-12 09:30:00-04:00     95.103313
2024-06-12 13:30:00-04:00     88.985669
2024-06-13 09:30:00-04:00     89.167977
2024-06-13 13:30:00-04:00     89.503191
2024-06-14 09:30:00-04:00     88.182612
2024-06-14 13:30:00-04:00     88.557969
2024-06-17 09:30:00-04:00     90.408551
2024-06-17 13:30:00-04:00     90.717578
2024-06-18 09:30:00-04:00     91.174